# Verifying `isotropy_metrics` by hand

Every metric is computed twice on one small layer `X` [N, d]:
1. with the real classes (`build_metrics` + `compute_layer`, the same path `run.py` uses), and
2. from scratch with plain NumPy loops that follow the paper formulas directly (no shortcuts).

The two must agree to ~1e-6. At the end, a few point clouds with known answers check that the values also make sense.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

# The kernel starts in the notebook's folder (tests/), so its parent is A2/
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import torch

from isotropy_metrics import LayerStats, build_metrics, compute_layer
from registry import METRICS

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.precision", 4)

## 1. A small layer

N = 10 points in d = 3 dims from 3 texts. The cloud is stretched along one axis (so it is *not* isotropic),
and row 9 is an exact copy of row 0. This copies what happens at layer 0, where the same token id gives the
same vector, and it exercises the duplicate removal in the ID metric.

`K` is the number of neighbours for the ID estimator. It has to be smaller than the number of distinct points, so it is 3 here instead of 20.

In [3]:
N, d, K = 10, 3, 3
rng = np.random.default_rng(0)
X_np = rng.normal(size=(N, d)) * np.array([3.0, 1.0, 0.3]) + np.array([1.0, -2.0, 0.5])
X_np[9] = X_np[0]                                  # exact duplicate point
text_ids_np = np.array([0, 0, 0, 1, 1, 1, 1, 2, 2, 2])

# float32 on the way in, as in run.py; the NumPy reference uses the same rounded values in float64
X = torch.tensor(X_np, dtype=torch.float32)
X_np = X.double().numpy()
text_ids = torch.tensor(text_ids_np)

pd.DataFrame(X_np, columns=[f"x{j}" for j in range(d)]).assign(text=text_ids_np)

,x0,x1,x2,text
0,1.3772,-2.1321,0.6921,0
1,1.3147,-2.5357,0.6085,0
2,4.9120,-1.0529,0.2889,0
3,-2.7963,-2.6233,0.5124,1
4,-5.9751,-2.2188,0.1262,1
5,-1.1968,-2.5443,0.4051,1
6,2.2349,-0.9575,0.4614,1
7,5.0994,-2.6652,0.6055,2
8,3.7104,-1.9060,0.2770,2
9,1.3772,-2.1321,0.6921,2


## 2. What the library returns

In [4]:
metrics = build_metrics(METRICS.names(), {"id": {"k": K}})
print("metrics:", [type(m).__name__ for m in metrics])
ours = compute_layer(metrics, X, text_ids)
pd.Series(ours, name="library")

metrics: ['IsoScore', 'MaxExplainableVariance', 'AvgRandomCosine', 'IntrinsicDimension']


isoscore       0.0294
mev            0.9715
avg_cos        0.3567
id_mle         3.0491
id_score       1.0164
id_n_unique    9.0000
Name: library, dtype: float64

## 3. The same numbers from scratch

### 3a. Covariance eigenvalues (these feed IsoScore and MEV)
`LayerStats.eigvals` should equal the eigenvalues of the sample covariance (denominator N-1).

In [5]:
C = np.cov(X_np.T)                      # [d, d], rows of X_np.T = variables
eig_ref = np.linalg.eigvalsh(C)         # ascending
eig_lib = LayerStats(X, text_ids).eigvals.numpy()
print("covariance:\n", C)
print("eigvals ref:", eig_ref)
print("eigvals lib:", eig_lib)
print("max |diff| :", np.abs(eig_ref - eig_lib).max())

covariance:
 [[12.1392  0.816   0.2115]
 [ 0.816   0.3812 -0.0378]
 [ 0.2115 -0.0378  0.0367]]
eigvals ref: [ 0.0241  0.3339 12.1991]
eigvals lib: [ 0.0241  0.3339 12.1991]
max |diff| : 1.1102230246251565e-16


### 3b. MEV: variance explained by the first principal component
$$\text{MEV} = \frac{\lambda_{\max}}{\sum_i \lambda_i}, \qquad \text{range } [1/d,\ 1]$$

In [6]:
mev_ref = eig_ref.max() / eig_ref.sum()
print(f"mev ref = {mev_ref:.6f}   lib = {ours['mev']:.6f}   (1/d = {1/d:.4f})")

mev ref = 0.971494   lib = 0.971494   (1/d = 0.3333)


### 3c. IsoScore (Rudman et al., 2022), the paper's steps one at a time
1. Rotate the points onto their principal components (PCA).
2. Take the variance vector $\Sigma_D$ of the rotated points.
3. Rescale it to length $\sqrt{d}$: $\hat\Sigma_D$.
4. Isotropy defect $\delta = \lVert\hat\Sigma_D - \mathbf 1\rVert / \sqrt{2(d-\sqrt d)}$.
5. Fraction of dims used $k = (d - \delta^2 (d-\sqrt d))^2 / d^2$.
6. $\text{IsoScore} = (d\,k - 1)/(d - 1)$.

Here the PCA is done explicitly on the points, while the library uses covariance eigenvalues. This makes the two computations independent.

In [7]:
Xc = X_np - X_np.mean(0)
_, _, Vt = np.linalg.svd(Xc, full_matrices=False)
X_pca = Xc @ Vt.T                                    # step 1
var = X_pca.var(axis=0, ddof=1)                      # step 2
var_hat = var * np.sqrt(d) / np.linalg.norm(var)     # step 3
delta = np.linalg.norm(var_hat - 1) / np.sqrt(2 * (d - np.sqrt(d)))   # step 4
k_frac = (d - delta**2 * (d - np.sqrt(d)))**2 / d**2                  # step 5
iso_ref = (d * k_frac - 1) / (d - 1)                                  # step 6
print("variance per PC:", var, "| normalised:", var_hat)
print(f"defect = {delta:.4f}, fraction of dims used = {k_frac:.4f}")
print(f"isoscore ref = {iso_ref:.6f}   lib = {ours['isoscore']:.6f}")

# Official package, if installed (pip install IsoScore)
try:
    from IsoScore import IsoScore as official
    print(f"isoscore official package = {float(official.IsoScore(X_np)):.6f}")
except ImportError:
    print("IsoScore package not installed, skipping")

variance per PC: [12.1991  0.3339  0.0241] | normalised: [1.7314 0.0474 0.0034]
defect = 0.9800, fraction of dims used = 0.3529
isoscore ref = 0.029375   lib = 0.029375
isoscore official package = 0.029375


### 3d. Average cosine between tokens of *different* texts
The library uses a sum-of-vectors shortcut. The check below loops over every pair explicitly.
In the matrix, entries between tokens of the same text are blanked out because they are not counted.

In [8]:
U = X_np / np.linalg.norm(X_np, axis=1, keepdims=True)
cos = U @ U.T
vals = [cos[i, j] for i in range(N) for j in range(N) if text_ids_np[i] != text_ids_np[j]]
avgcos_ref = float(np.mean(vals))
print(f"{len(vals)} ordered cross-text pairs")
print(f"avg_cos ref = {avgcos_ref:.6f}   lib = {ours['avg_cos']:.6f}")

same_text = text_ids_np[:, None] == text_ids_np[None, :]
pd.DataFrame(np.where(same_text, np.nan, cos)).round(3)

66 ordered cross-text pairs
avg_cos ref = 0.356674   lib = 0.356674


,0,1,2,3,4,5,6,7,8,9
0,NaN,NaN,NaN,0.206,-0.203,0.543,0.835,0.862,0.852,1.000
1,NaN,NaN,NaN,0.291,-0.116,0.618,0.781,0.819,0.809,0.994
2,NaN,NaN,NaN,-0.556,-0.841,-0.216,0.973,0.963,0.966,0.696
3,0.206,0.291,-0.556,NaN,NaN,NaN,NaN,-0.311,-0.324,0.206
4,-0.203,-0.116,-0.841,NaN,NaN,NaN,NaN,-0.664,-0.672,-0.203
5,0.543,0.618,-0.216,NaN,NaN,NaN,NaN,0.056,0.044,0.543
6,0.835,0.781,0.973,NaN,NaN,NaN,NaN,0.994,0.990,0.835
7,0.862,0.819,0.963,-0.311,-0.664,0.056,0.994,NaN,NaN,NaN
8,0.852,0.809,0.966,-0.324,-0.672,0.044,0.990,NaN,NaN,NaN
9,1.000,0.994,0.696,0.206,-0.203,0.543,0.835,NaN,NaN,NaN


### 3e. Intrinsic dimension (Levina & Bickel MLE)
For every distinct point, let $T_1 \le \dots \le T_k$ be the distances to its k nearest neighbours. Then
$$\hat m = \frac{k-1}{\sum_{j=1}^{k-1} \log(T_k / T_j)}, \qquad \text{id\_mle} = \text{mean of } \hat m, \qquad \text{id\_score} = \text{id\_mle}/d$$
Duplicates are removed first, so only 9 of the 10 points are used.

In [9]:
P = np.unique(X_np, axis=0)
n_u = len(P)
rows = []
for i in range(n_u):
    dist = np.sort([np.linalg.norm(P[i] - P[j]) for j in range(n_u) if j != i])[:K]
    m_i = (K - 1) / np.sum(np.log(dist[-1] / dist[:-1]))
    rows.append({"point": i, **{f"T{j+1}": t for j, t in enumerate(dist)}, "m_hat": m_i})
per_point = pd.DataFrame(rows)
id_ref = per_point["m_hat"].mean()
print(f"unique points: ref = {n_u}   lib = {ours['id_n_unique']}")
print(f"id_mle   ref = {id_ref:.6f}   lib = {ours['id_mle']:.6f}")
print(f"id_score ref = {id_ref / d:.6f}   lib = {ours['id_score']:.6f}")
per_point

unique points: ref = 9   lib = 9
id_mle   ref = 3.049075   lib = 3.049076
id_score ref = 1.016358   lib = 1.016359


,point,T1,T2,T3,m_hat
0,0,3.2276,4.7975,7.3126,1.6137
1,1,1.6050,3.2276,4.1130,1.6900
2,2,1.6050,2.5197,2.6225,3.7665
3,3,0.4169,1.8328,2.4992,0.9519
4,4,0.4169,1.4726,2.3806,0.8998
5,5,1.4726,1.7638,1.8328,7.7771
6,6,1.4737,1.6167,1.7638,7.4967
7,7,1.4737,1.6537,2.6844,1.8448
8,8,1.6167,1.6537,3.3380,1.4012


In [1]:
import torch
i,j = torch.randint(3, 10, (2, 2))
print(i,j)

tensor([7, 8]) tensor([8, 5])


## 4. Summary: library vs from scratch

In [10]:
ref = {"isoscore": iso_ref, "mev": mev_ref, "avg_cos": avgcos_ref,
       "id_mle": id_ref, "id_score": id_ref / d, "id_n_unique": n_u}
summary = pd.DataFrame({"library": pd.Series(ours), "from scratch": pd.Series(ref)})
summary["abs diff"] = (summary["library"] - summary["from scratch"]).abs()
summary["ok"] = summary["abs diff"] < 1e-5
summary

,library,from scratch,abs diff,ok
isoscore,0.0294,0.0294,0.0000e+00,True
mev,0.9715,0.9715,0.0000e+00,True
avg_cos,0.3567,0.3567,1.1102e-16,True
id_mle,3.0491,3.0491,2.2983e-07,True
id_score,1.0164,1.0164,7.6610e-08,True
id_n_unique,9.0000,9.0000,0.0000e+00,True


## 5. Point clouds with known answers
A match with the from-scratch code shows the formulas are implemented correctly. These cases check that the values also mean what they should:

| cloud | isoscore | mev | avg_cos | id_mle |
|---|---|---|---|---|
| isotropic Gaussian, centred | ~1 | ~1/d | ~0 | ~d |
| same Gaussian, shifted far from the origin | ~1 (shift-invariant) | ~1/d | ~1 (all point the same way) | ~d |
| rank one (a line through the origin) | ~0 | ~1 | ~0 (each pair is ±1; the two signs cancel) | ~1 |
| a 2-D plane inside d dims | (2-1)/(d-1) = 0.111 for d=10 (2 of 10 dims used) | ~0.5 | ~0 | ~2 |

ID tends to come out a little low when the true dimension is high, so for the Gaussian it may land slightly below d. In the tiny 9-point example above, ID is only a check of the arithmetic: 9 points are far too few for a meaningful estimate.

In [11]:
def run_case(X, n_texts=100):
    """Metrics for an [N, d] array, tokens spread round-robin over n_texts texts."""
    X = torch.tensor(X, dtype=torch.float32)
    tids = torch.arange(X.shape[0]) % n_texts
    return compute_layer(build_metrics(METRICS.names(), {"id": {"k": 20}}), X, tids)

rng = np.random.default_rng(1)
Nb, db = 5000, 10
gauss = rng.normal(size=(Nb, db))
cases = {
    "isotropic gaussian":     gauss,
    "gaussian + big shift":   gauss + 50,
    "rank one (line)":        rng.normal(size=(Nb, 1)) * rng.normal(size=(1, db)),
    "2-D plane in 10-D":      rng.normal(size=(Nb, 2)) @ np.linalg.qr(rng.normal(size=(db, db)))[0][:2],
}
pd.DataFrame({name: run_case(Xc) for name, Xc in cases.items()}).T.assign(**{"1/d": 1 / db})

,isoscore,mev,avg_cos,id_mle,id_score,id_n_unique,1/d
isotropic gaussian,9.9745e-01,0.1094,4.9740e-04,9.9575,0.9957,5000.0,0.1
gaussian + big shift,9.9745e-01,0.1094,9.9964e-01,9.9575,0.9957,5000.0,0.1
rank one (line),3.9475e-17,1.0000,-1.7293e-04,0.9513,0.0951,5000.0,0.1
2-D plane in 10-D,1.1109e-01,0.5049,4.7909e-05,2.1401,0.2140,5000.0,0.1
